In [0]:
spark.sql("USE CATALOG rio_dataengineering")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("USE SCHEMA bronze")

BASE_PATH = "abfss://people-data@peopleanalyticslakedomma.dfs.core.windows.net/bronze"

TABLES = [
    "employees",
    "departments",
    "locations",
    "payroll",
    "training",
    "engagement",
    "leave"
]

print("Bronze configuration ready.")

In [0]:
def ingest_csv_to_bronze(table_name: str) -> int:

    source_path = f"{BASE_PATH}/{table_name}.csv"
    target_table = f"rio_dataengineering.bronze.{table_name}"

    df = (
        spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(source_path)
    )

    row_count = df.count()

    (
        df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
    )

    return row_count

In [0]:
load_results = {}

for table_name in TABLES:

    row_count = ingest_csv_to_bronze(table_name)

    load_results[table_name] = row_count

    print(f"✅ Loaded bronze.{table_name}: {row_count} rows")

In [0]:
validation = []

for table_name in TABLES:

    bronze_rows = spark.table(
        f"rio_dataengineering.bronze.{table_name}"
    ).count()

    validation.append(
        (
            table_name,
            load_results[table_name],
            bronze_rows,
            "PASS" if load_results[table_name] == bronze_rows else "FAIL"
        )
    )

validation_df = spark.createDataFrame(
    validation,
    ["table", "source_rows", "bronze_rows", "status"]
)

display(validation_df)